# deployment — Wrap the vol forecaster in a stateless predict_one

## Project goal

Take the LightGBM vol forecaster (trained or loaded) and package it for production: stateless `predict_one(row)` with feature-name reorder, Pydantic schema generated from `feature_names`, structured JSON logs, async wrapper, property test for ordering invariance, and minimal `/health` + `/ready` stubs.


## Why this exercises the cheatsheet

Bridges the gap between 'I have a model' and 'I have a service'. Every step lives in this notebook (no separate FastAPI server) — uses TestClient where needed. Forces the deployment cheatsheet's full surface area: bundle, schema, logging, async, ops endpoints.


## Sub-task 1: Train + bundle

Train the model (or reuse). Save a bundle with `model`, `feature_names`, `threshold` (use 0.0 for the regression case, or pick a meaningful vol-spike threshold), `trained_through`, and `model_version`.

**Patterns this forces:**

- `joblib.dump({...}, path)`
- bundle metadata fields
- version string


In [ ]:
# Your answer here

## Sub-task 2: predict_one helper

Implement `predict_one(row: dict) -> dict` that loads the bundle, reorders the row by `feature_names`, and returns `{prob_up_or_value, model_version, trained_through}`. The reorder is the single critical line.

**Patterns this forces:**

- `pd.DataFrame([row])[art['feature_names']]` ← THE critical reorder
- load bundle inside the function (stateless)
- return a dict with metadata


In [ ]:
# Your answer here

## Sub-task 3: Pydantic schema from feature_names

Generate the input schema dynamically: `create_model('PredictRequest', **{f: (float, ...) for f in feature_names})`. Validate a known-good payload AND a payload missing one feature — confirm the missing one raises `ValidationError`.

**Patterns this forces:**

- `pydantic.create_model(...)` for runtime schemas
- `(float, ...)` tuple = required field
- catch `ValidationError` to demo the failure


In [ ]:
# Your answer here

## Sub-task 4: Property test: order invariance

Write `test_predict_one_order_invariance(n=20)` that draws random rows from val, calls `predict_one` with the row's keys in original order and reversed order, asserts the predictions match within 1e-9. Should pass because of the reorder; if you remove the reorder, it should fail.

**Patterns this forces:**

- property-based test pattern
- `dict(reversed(list(row.items())))` for permutation
- abs-diff tolerance check


In [ ]:
# Your answer here

## Sub-task 5: Structured JSON logging

Configure a logger with a `JSONFormatter` and a per-call correlation ID via `uuid.uuid4().hex`. Every prediction logs the request_id, input features, and the output value. Confirm the log line is valid JSON by parsing it back.

**Patterns this forces:**

- custom `logging.Formatter` returning JSON
- correlation ID via `uuid4().hex`
- `extra={'extra': {...}}` for structured fields


In [ ]:
# Your answer here

## Sub-task 6: Async wrapper

Define `async def predict_one_async(row)` that wraps the sync version in `asyncio.get_event_loop().run_in_executor(None, ...)`. Call it from an `asyncio.run(...)` block. Comment: when does the async version add value vs the sync one (hint: only when used in an async context)?

**Patterns this forces:**

- `asyncio.get_event_loop().run_in_executor(None, fn, arg)`
- `asyncio.run(...)` for top-level demo
- comment on when async is right vs sync


In [ ]:
# Your answer here

## Sub-task 7: Ops endpoints (FastAPI + TestClient)

Build a tiny FastAPI app with `/health` (always 200) and `/ready` (503 if model not loaded, 200 with model metadata if loaded). Drive with `TestClient`. Confirm `/health` returns 200 immediately; `/ready` returns 503 before model load and 200 after.

**Patterns this forces:**

- minimal FastAPI app with two routes
- `@app.on_event('startup')` to load model — and call it directly when using bare `TestClient` without `with` context
- `response.status_code = 503` for not-ready


In [ ]:
# Your answer here

## What success looks like

- A self-contained notebook that ends with: bundle saved, predict_one verified, schema validated, property test passing, structured logs JSON-parseable, async demoed once, /health + /ready returning correct codes.
- Zero hardcoded paths — everything via `os.environ.get(...)` or the bundle.
- The property test would catch the feature-ordering bug if you regressed it.
- You'd be willing to drop the relevant cells into `app/main.py` and ship.
